# 让 Claude 进入"JSON 模式"

Claude 并没有正式的"JSON 模式"，也没有受约束的采样。但不用担心——你仍然可以从 Claude 获得可靠的 JSON！本示例将展示如何实现。

首先，看看 Claude 的默认行为。

In [ ]:
%pip install anthropic

In [23]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Endpoint:", os.environ.get("ANTHROPIC_BASE_URL"))
print("Gateway key configured:", bool(os.environ.get("ANTHROPIC_API_KEY")))

expected = Path(r"D:\tutorial\claude-cookbooks\.venv\Scripts\python.exe")
if Path(sys.executable).resolve() != expected.resolve():
    raise RuntimeError("请切换到 cookbook 项目内核。")

Python: D:\tutorial\claude-cookbooks\.venv\Scripts\python.exe
Endpoint: https://qwen.smjtools.com
Gateway key configured: True


In [24]:
import json
import re
from pprint import pprint

from anthropic import Anthropic

In [ ]:
# client = Anthropic()
# MODEL_NAME = "claude-opus-4-8"

In [3]:
import os

gateway_url = os.environ.get("ANTHROPIC_BASE_URL", "").rstrip("/")
gateway_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()

if gateway_url != "https://qwen.smjtools.com":
    raise RuntimeError("当前内核没有 cookbook 网关配置。")
if not gateway_key or gateway_key == "YOUR_QWEN_GATEWAY_KEY":
    raise RuntimeError("请配置个人网关密钥并重启 Jupyter。")
if gateway_key.startswith("sk-ant-"):
    raise RuntimeError("不要将 Anthropic 密钥发送到 Qwen 网关。")
if os.environ.get("ANTHROPIC_AUTH_TOKEN"):
    raise RuntimeError("存在额外 auth token，请使用独立启动环境。")

client = Anthropic(
    api_key=gateway_key,
    base_url=gateway_url,
    timeout=180.0,
    max_retries=0,
)
MODEL_NAME = "strata"

In [4]:
message = (
    client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": "Give me a JSON dict with names of famous athletes & their sports.",
            },
        ],
    )
    .content[0]
    .text
)
print(message)

```json
{
  "Michael Jordan": "Basketball",
  "Usain Bolt": "Track and Field",
  "Serena Williams": "Tennis",
  "Cristiano Ronaldo": "Football (Soccer)",
  "Simone Biles": "Gymnastics",
  "Michael Phelps": "Swimming",
  "LeBron James": "Basketball",
  "Lionel Messi": "Football (Soccer)",
  "Tiger Woods": "Golf",
  "Muhammad Ali": "Boxing"
}
```


Claude 遵循了指令，输出了一个漂亮的字典，我们可以用代码提取它：

In [5]:
def extract_json(response):
    json_start = response.index("{")
    json_end = response.rfind("}")
    return json.loads(response[json_start : json_end + 1])


extract_json(message)

{'Michael Jordan': 'Basketball',
 'Usain Bolt': 'Track and Field',
 'Serena Williams': 'Tennis',
 'Cristiano Ronaldo': 'Football (Soccer)',
 'Simone Biles': 'Gymnastics',
 'Michael Phelps': 'Swimming',
 'LeBron James': 'Basketball',
 'Lionel Messi': 'Football (Soccer)',
 'Tiger Woods': 'Golf',
 'Muhammad Ali': 'Boxing'}

以下说法不再适用：
> 但如果我们希望 Claude 跳过开场白，直接输出 JSON 呢？一个简单的办法是预先填入 Claude 的响应，并包含一个 "{" 字符。

In [16]:
message = (
    client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": "Give me a JSON dict with names of famous athletes & their sports.",
            },
            # {"role": "assistant", "content": "Here is the JSON requested:\n{"},
        ],
    )
    .content[0]
    .text
)
print(message)

```json
{
  "Michael Jordan": "Basketball",
  "Usain Bolt": "Track and Field",
  "Serena Williams": "Tennis",
  "Cristiano Ronaldo": "Football (Soccer)",
  "Simone Biles": "Gymnastics",
  "Tiger Woods": "Golf",
  "Michael Phelps": "Swimming",
  "LeBron James": "Basketball",
  "Lionel Messi": "Football (Soccer)",
  "Katie Ledecky": "Swimming"
}
```


现在我们要做的只是把预先填入的 "{" 加回来，然后就可以提取 JSON 了。

In [18]:
# output_json = json.loads("{" + message[: message.rfind("}") + 1])
output_json = extract_json(message)
output_json

{'Michael Jordan': 'Basketball',
 'Usain Bolt': 'Track and Field',
 'Serena Williams': 'Tennis',
 'Cristiano Ronaldo': 'Football (Soccer)',
 'Simone Biles': 'Gymnastics',
 'Tiger Woods': 'Golf',
 'Michael Phelps': 'Swimming',
 'LeBron James': 'Basketball',
 'Lionel Messi': 'Football (Soccer)',
 'Katie Ledecky': 'Swimming'}

对于很长、很复杂的提示词，其中包含多个 JSON 输出，以致于用字符串搜索 "{" 和 "}" 无法奏效时，你也可以让 Claude 把每个 JSON 项输出在指定的标签内，便于后续提取。

In [19]:
message = (
    client.messages.create(
        model=MODEL_NAME,
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": """Give me a JSON dict with the names of 5 famous athletes & their sports.
Put this dictionary in <athlete_sports> tags.

Then, for each athlete, output an additional JSON dictionary. In each of these additional dictionaries:
- Include two keys: the athlete's first name and last name.
- For the values, list three words that start with the same letter as that name.
Put each of these additional dictionaries in separate <athlete_name> tags.""",
            },
            # {"role": "assistant", "content": "Here is the JSON requested:"},
        ],
    )
    .content[0]
    .text
)
print(message)

<athlete_sports>
{
  "Michael Jordan": "Basketball",
  "Serena Williams": "Tennis",
  "Usain Bolt": "Track and Field",
  "Cristiano Ronaldo": "Soccer",
  "Simone Biles": "Gymnastics"
}
</athlete_sports>

<athlete_name>
{
  "Michael": ["Mountain", "Momentum", "Majesty"],
  "Jordan": ["Journey", "Justice", "Jubilee"]
}
</athlete_name>

<athlete_name>
{
  "Serena": ["Silence", "Strength", "Serenity"],
  "Williams": ["Wisdom", "Wonder", "Willpower"]
}
</athlete_name>

<athlete_name>
{
  "Usain": ["Unity", "Uplift", "Universe"],
  "Bolt": ["Bravery", "Balance", "Bloom"]
}
</athlete_name>

<athlete_name>
{
  "Cristiano": ["Courage", "Clarity", "Calm"],
  "Ronaldo": ["Resilience", "Radiance", "Resolve"]
}
</athlete_name>

<athlete_name>
{
  "Simone": ["Spirit", "Sincerity", "Simplicity"],
  "Biles": ["Brilliance", "Boldness", "Beauty"]
}
</athlete_name>


现在，我们可以用提取用的正则表达式取得所有字典。

In [20]:
import re


def extract_between_tags(tag: str, string: str, strip: bool = False) -> list[str]:
    ext_list = re.findall(f"<{tag}>(.+?)</{tag}>", string, re.DOTALL)
    if strip:
        ext_list = [e.strip() for e in ext_list]
    return ext_list


athlete_sports_dict = json.loads(extract_between_tags("athlete_sports", message)[0])
athlete_name_dicts = [json.loads(d) for d in extract_between_tags("athlete_name", message)]

In [21]:
pprint(athlete_sports_dict)

{'Cristiano Ronaldo': 'Soccer',
 'Michael Jordan': 'Basketball',
 'Serena Williams': 'Tennis',
 'Simone Biles': 'Gymnastics',
 'Usain Bolt': 'Track and Field'}


In [22]:
pprint(athlete_name_dicts, width=1)

[{'Jordan': ['Journey',
             'Justice',
             'Jubilee'],
  'Michael': ['Mountain',
              'Momentum',
              'Majesty']},
 {'Serena': ['Silence',
             'Strength',
             'Serenity'],
  'Williams': ['Wisdom',
               'Wonder',
               'Willpower']},
 {'Bolt': ['Bravery',
           'Balance',
           'Bloom'],
  'Usain': ['Unity',
            'Uplift',
            'Universe']},
 {'Cristiano': ['Courage',
                'Clarity',
                'Calm'],
  'Ronaldo': ['Resilience',
              'Radiance',
              'Resolve']},
 {'Biles': ['Brilliance',
            'Boldness',
            'Beauty'],
  'Simone': ['Spirit',
             'Sincerity',
             'Simplicity']}]


总结一下：

- 你可以用字符串解析提取 "```json" 和 "```" 之间的文本，从而得到 JSON。
- 你可以通过一条不完整的 Assistant 消息去掉 JSON *之前*的开场白。（但这样就无法让 Claude 先做"思维链"来提升智能程度再开始输出 JSON。）
- 你可以使用停止序列去掉 JSON *之后*的文本。
- 你可以指示 Claude 把 JSON 输出在 XML 标签内，以便在更复杂的提示词中方便地收集。